In [ ]:
import os # 환경 변수 설정에 사용됨
import re # 정규 표현식 처리 모듈임. ReAct에이전트의 출력에서 Action input, Final Answer 등을 추출하기 위해 사용함.
import io # 문자열을 파일 처럼 다룰수 있게 해주는 모듈임. StringIO를 사용해 에이전트의 실행 로그를 메모리상에서 캡처함.
import requests # HTTP 요청을 보내는 라이브러리임. PDF 파일을 url에서 다운로드 받을 때 사용함.
from dotenv import load_dotenv # 환경 변수 외부 설정에 사용됨.
from typing import Dict, List, Optional, Tuple # 함수의 매개변수나 반환값에 타입 힌트를 제공하는 모듈임. 코드 가독성을 높임.
from contextlib import redirect_stdout #컨텍스트 관리 유틸리티를 제공하는 모듈임. redirect_stdout를 사용해 에이전트 실행 중 출력을 다른 곳으로 보낼 수 있음.

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader # langchain_community 는 커뮤니티에서 개발한 랭체인 확장 기능들을 제공함.
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool #langchain_core는 랭체인의 핵심 기능. create_retriever_tool은 검색기를 에이전트가 사용할 수있는 도구 형태로 변환.
from langchain_core.prompts import PromptTemplate # PromptTemplate는 프롬프트를 동적으로 생성하는 데 사용함.

import gradio as gr # 머신 러닝 모델을 위한 웹 인터페이스를 쉽고 빠르게 구축할 수있는 라이브러리임. 코드 몇 줄로 전문적인 채팅 데모 사이트를 만들 수 있음.

#os.environ["OPEN_API_KEY"]= ""
load_dotenv()

# 실습 데이터 다운로드.
"""
urls = [
    "https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/ict_japan_2024.pdf",
	"https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/ict_usa_2024.pdf",
	"https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/blockchain_usa_2025.pdf"
]
"""

# 각 파일을 다운로드.
"""
for url in urls :
    filename = url.split("/")[-1] # URL에서 파일명 추출
    response = requests.get(url)

    with open(filename, "wb") as f:  # with문은 "시작할때 이거 해두고, 끝날 때 무슨일이 있더라도 정리해줘" 를 보장해 주는 스마트한 문법임.
        f.write(response.content)

    print(f"{filename} 다운로드 완료")    
"""

# 임베딩 설정.
embd = OpenAIEmbeddings()

def create_pdf_retriever(
        pdf_path: str,                      # PDF 파일 경로
        persist_directory: str,             # 벡터 스토어 저장 경로(persist : 지속)
        embedding_model: OpenAIEmbeddings,  # OpenAIEmbeddings 임베딩 모듈
        chunk_size: int=512,                # 청크 크기 기본값 : 512
        chunk_overlap: int=0                # 청크 오버랩 크기 기본값 : 0
) -> Chroma.as_retriever:                                                   # 타입 힌트(Type Hint) : 파이썬은 원래 함수의 타입을 미리 정하지 않는 동적 타이핑 언어임
                                                                            # 하지만 코드의 가독성을 높이기 위해 def 함수이름(...) -> 반환타입: 의 형태로 명시할 수 있음.
    # PDF 파일 로드
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    # chunking
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    doc_splits = text_splitter.split_documents(data)

    # 벡터 스토어로 적재
    vectorstore = Chroma.from_documents(
        persist_directory = persist_directory,
        documents = doc_splits,
        embedding = embedding_model
    )

    return vectorstore.as_retriever() # retriever 객체는 자연어 질의에 대해 관련성 높은 PDF 내용을 검색할 수있는 검색기임.

# 주제별로 독립적인 벡터 검색기를 생성함.
# 일본 ICT 정책 데이터베이스 생성.
retriever_ict_japan = create_pdf_retriever(
    pdf_path = "ict_japan_2024.pdf",
    persist_directory = "db_ict_policy_japan_2024",
    embedding_model = embd
)

# 미국 ICT 정책 데이터베이스 생성.
retriever_ict_usa = create_pdf_retriever(
    pdf_path = "ict_usa_2024.pdf",
    persist_directory = "db_ict_policy_usa_2024",
    embedding_model = embd
)

# 미국 블록체인 동향 데이터베이스 생성
retriever_blockchain_usa  = create_pdf_retriever(
    pdf_path = "blockchain_usa_2025.pdf",
    persist_directory = "db_blockchain_usa_2025",
    embedding_model = embd
)

# create_pdf_retriever 함수를 사용해 각각의 벡터 데이터베이스 검색기를 생성함.
# persist_directory 에 db_ict_policy_japan_2024, db_ict_policy_usa_2024, db_blockchain_usa_2025 라는 서로 다른 디렉토리 경로를 지정하면 
# 세개의 벡터 데이터베이스가 서로 영향을 주지 않고, 독립적으로 저장되고 관리함.

# 만약 같은 경로를 사용했다면 두번째 벡터 데이터베이스를 처리할 때 첫번째 벡터 데이터베이스를 덮어쓰거나 혼합될 수 있기 때문에 이렇게 경로를 분리하는 것이 중요함.

# 결론적으로 일본 ICT 정책을 다루는 ict_japan_2024.pdf 파일을 열어 db_ict_policy_japan_2024 디렉터리에 벡터 데이터베이스를 생성함.
# 마찬가지로 미국 ICT 정책을 다루는 ict_usa_2024.pdf 파일을 열어 db_ict_policy_usa_2024 디렉터리에,
# 미국 블록체인 동향에 대해 다루는 blockchain_usa_2025.pdf 파일을 db_blockchain_usa_2025 디렉터리에 벡터 데이터 베이스로 저장해 세개의 검색기를 만들었음.

# 이제 이렇게 만들어진 3개의 retriever 객체를 create_retriever_tool 함수를 통해 ReAct 에이전트가 사용할 수 있는 검색 도구 형태로 변환해 봄.
ict_japan_engine = create_retriever_tool(
    retriever = retriever_ict_japan,
    name = "japan_ict_trend_searcher",
    description = "일본의 ICT 산업의 시장 동향 정보를 제공합니다."
)

ict_usa_engine = create_retriever_tool(
    retriever = retriever_ict_usa,
    name = "usa_ict_trend_searcher",
    description = "미국의 ICT 산업의 시장 동향 정보를 제공합니다."
)

blockchain_usa_engine = create_retriever_tool(
    retriever = retriever_blockchain_usa,
    name = "usa_blockchain_trend_searcher",
    description = "미국의 블록체인 산업의 동향 정보를 제공합니다."
)

tools = [ict_japan_engine, ict_usa_engine, blockchain_usa_engine]
tool_map: Dict[str, object] = {t.name: t for t in tools}

# create_retriever_tool 함수는 retriever 객체를 ReAct 에이전트가 활용할 수 있는 형태의 도구로 변환하는 함수임.
# 앞에서 생성한 retriever_ict_japan 등의 객체를 바탕으로 도구의 이름(name)과 설명(description)을 추가해 ReAct 메이전트가 활용할 수 있는 도구로 만듬.

# 이때 도구를 생성할때 description 에는 각 검색기의 상세한 용도를 작성해야 함.
# 예를 들어, 일본 ICT 검색기의 경우, '일본의 ICT 시장 동향 정보를 제공합니다.'라는 설명을 작성했음.
# 이후 ReAct 에이전트가 동작할때 에이전트는 여기서 적힌 설명을 보고 사용자의 질문에 따라 도구를 선택함.
# 즉, 에이전트는 사용자가 미국의 ICT 산업과 관련된 질문을 하면 usa_ict라는 이름의 도구를 사용하고 일본의 ICT 산업과 관련된 질문을 하면 japan_ict 라는 이름의 도구를 사용하게 됨.
# 그리고 세개의 주제와 상관이 없는 질문에는 어떠한 도구도 사용하지 않고 답변을 하게 함.
# 이는 에이전트가 오로지 도구의 description을 보고 판단함. 따라서 description에 도구에 대해 설명이 제대로 적혀 있지 않다면 에이전트는 상황에 맞는 도구 선택을 제대로 할수 없으므로
# 상세한 설명을 기재해야 함.

# 생성된 세걔의 도구인 ict_japan_engine , ict_usa_engine, blockchain_usa_engine 을 tool 리스트에 담음.
# 이 tools 리스트를 뒤에 코드에서 ReAct 에이전트에 전달하고, 에이전트는 이 도구들의 description 을 통해 각 질문에 가장 적합한 도구를 선택하고 정확한 검색을 수행할 수 있게 함.

# tool_map 은 도구이 이름을 키(key), 도구 객체를 값(value)으로 답는 딕셔너리임.
# 이렇게 만들어 두면 에이전트가 "japan_ict_trend_searcher"라는 도구를 사용하겠다고 결정했을때 tool_map["japan_ict_trend_searcher"]로 해당 도구를 즉시 찾아서 실행할 수 있음.

# 이제 ReAct 에이전트가 사용할 프롬프트 템플릿을 설정하겠음.
# ReAct 에이전트는 정해진 형식에 따라 사고하고 행동해야 하므로 이 형식을 명확하게 지시하는 프롬프트가 필요함.
react_template = '''
다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:
{tools}

다음 형식을 사용하세요:
Question: 답변을 해야하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
...(이 Thought/Action/Action Input/Observation 의 과정이 N번 반복할 수 있습니다.)
Thought: 이제 최종 답변을 알겠습니다.
Final Answer: 원랙 입력된 질문에 대한 최종 답변

## 추가적인 주의사항
- 반드시 [Thought->Action->Action Input format] 이 사이클의 순서를 준수하십시오. 항상 Action 전에는 Thought 가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것이 중요합니다.
- 정보가 취합 되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question : {input}
{agent_scratchpad} 
'''
# scratchpad :고속 작업용 보조기억장치
# 실제 상황에서 {input} 부분에는 사용자가 입력한 질문이 그대로 들어감. 예를 들어, '일본의 AI 정책에 대해 알려줘'라고 질문하면 이 텍스트가 Question: 뒤에 입력됨.
# {agent_scratchpad} 부분은 에이전트가 사고 과정에 해당하는 ReAct 사이클이 지속적으로 누적되는 공간임.
# 처음에는 비어 있지만 에이전트가 실행되면서 Thought, Action, Action Input, Observation 이라는 react 사이클이 Final Answer 단계에 이르기까지 계속 누적되어 쌓임.
prompt = PromptTemplate.from_template(react_template)

# 이제 프롬프트 템플릿에 실제 값들을 채워 넣는 함수들을 구현하겠음.
def _format_tools_for_prompt(ts: List[object]) -> Tuple[str, str]:
    lines, names = [], []

    for t in ts:
        names.append(t.name)
        desc = getattr(t, "description", "")
        lines.append(f"{t.name}: {desc}")

    return "\n".join(lines), ", ".join(names)

def _render_prompt(user_input: str, scratchpad: str) -> str:
    tools_str, tool_names = _format_tools_for_prompt(tools)

    return prompt.format(
        tools= tools_str,
        tool_name = tool_names,
        input = user_input,
        agent_scratchpad = scratchpad
    )

TypeError: object of type 'VectorStoreRetriever' has no len()